In [10]:
import qr
import importlib
importlib.reload(qr)

<module 'qr' from '/home/lexu/University/visible-light/inbedded/qr.py'>

In [11]:
import math
import numpy as np
import segno
import cv2
import moderngl

In [12]:
def create_qr(content):
    qr = segno.make_qr(content, version=10, mode="byte")
    matrix = qr.matrix
    arr = np.array(matrix, dtype=int)

    qr.save("output/qr.png", scale=10)

In [13]:
qcd = cv2.QRCodeDetector()

import zxingcpp

def read_qr(img):
    results = zxingcpp.read_barcodes(
        img,
        formats=zxingcpp.BarcodeFormat.QRCode
    )
    return [result.bytes for result in results]

In [4]:
from subtitle_manager import *

manager = SubtitleManager()
manager.import_from_srt("potato_chip_subs.srt")
manager.get_ready(1000)

[Subtitle(startTime=960, endTime=2720, sub_string="Judging by the number of cameras he's")]

In [8]:
%load_ext autoreload
%autoreload 2

import cv2
import segno
import numpy as np
from pathlib import Path
from protocol import *
from subtitle_manager import *
from video_processor import VideoProcessor

SUB_PATH = "potato_chip_subs.srt"
VIDEO_FILE = "potato_chip.mp4"
OUTPUT_PATH = "output/death_note.mp4"

vp = VideoProcessor(VIDEO_FILE, SUB_PATH)
vp.output(OUTPUT_PATH)



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Frame: 0, frame=<class 'NoneType'>, qr_signal=<class 'numpy.ndarray'>
Frame index: 0
Packet ID: 0
Previous packet ID: 0
QR signal: <class 'numpy.ndarray'>


AttributeError: 'NoneType' object has no attribute 'shape'

In [ ]:
import cv2
import numpy as np
import zxingcpp

# This should be the Irudin camera index
CAMERA_INDEX = 4

cap = cv2.VideoCapture(CAMERA_INDEX)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
cap.set(cv2.CAP_PROP_FPS, 30)

if not cap.isOpened():
    raise RuntimeError("Could not open camera")

decoder = MessageDecoder()
previous = None

print("Camera FPS:", cap.get(cv2.CAP_PROP_FPS))

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.resize(gray, (960, 540))

        if previous is not None:
            # Signed difference preserves bright/dark polarity
            diff = gray.astype(np.float32) - previous.astype(np.float32)

            # Threshold at zero to recover QR modules
            qr = np.where(diff > 0, 255, 0).astype(np.uint8)

            # Try both phase orders
            for candidate in (qr, cv2.bitwise_not(qr)):
                # Using this library because it returns bytes correctly unlike OpenCV
                results = zxingcpp.read_barcodes(
                    candidate,
                    formats=zxingcpp.BarcodeFormat.QRCode
                )

                for result in results:
                    for message in decoder.push(result.bytes):
                        print("Decoded:", message)

            cv2.imshow("Reconstructed QR", qr)

        previous = gray.copy()

        cv2.imshow("Camera", cv2.resize(frame, (960, 540)))

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()


Camera FPS: 30.0
Decoded: SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime=3500, sub_string='Hello World!')
Decoded: SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime=3500, sub_string='Hello World!')
Decoded: SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime=3500, sub_string='Hello World!')
Decoded: SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime=3500, sub_string='Hello World!')
Decoded: SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime=3500, sub_string='Hello World!')
Decoded: SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime=3500, sub_string='Hello World!')
Decoded: SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime=3500, sub_string='Hello World!')
Decoded: SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime=3500, sub_string='Hello World!')
Decoded: SendSubtitle(currentTime=1000, subtitleStartTime=1200, subtitleEndTime